In [2]:
import pandas as pd

df = pd.read_csv("suicide-watch/Suicide_Detection.csv")  # adjust name
print(df.head())
print(df.columns)

   Unnamed: 0                                               text        class
0           2  Ex Wife Threatening SuicideRecently I left my ...      suicide
1           3  Am I weird I don't get affected by compliments...  non-suicide
2           4  Finally 2020 is almost over... So I can never ...  non-suicide
3           8          i need helpjust help me im crying so hard      suicide
4           9  I’m so lostHello, my name is Adam (16) and I’v...      suicide
Index(['Unnamed: 0', 'text', 'class'], dtype='str')


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 232074 entries, 0 to 232073
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   Unnamed: 0  232074 non-null  int64
 1   text        232074 non-null  str  
 2   class       232074 non-null  str  
dtypes: int64(1), str(2)
memory usage: 5.3 MB


In [4]:
df = df.drop(columns=["Unnamed: 0"])
df = df.rename(columns={"text": "raw_text", "class": "label"})

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 232074 entries, 0 to 232073
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   raw_text  232074 non-null  str  
 1   label     232074 non-null  str  
dtypes: str(2)
memory usage: 3.5 MB


In [6]:
df.head()

,raw_text,label
0,Ex Wife Threatening SuicideRecently I left my ...,suicide
1,Am I weird I don't get affected by compliments...,non-suicide
2,Finally 2020 is almost over... So I can never ...,non-suicide
3,i need helpjust help me im crying so hard,suicide
4,"I’m so lostHello, my name is Adam (16) and I’v...",suicide


In [7]:
print(df["label"].value_counts())

label
suicide        116037
non-suicide    116037
Name: count, dtype: int64


In [8]:
df["length"] = df["raw_text"].apply(len)

df.groupby("label")["length"].mean()

label
non-suicide     329.218844
suicide        1050.060627
Name: length, dtype: float64

In [9]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["raw_text"].apply(clean_text)

In [10]:
keywords = ["kill myself", "end my life", "suicide", "hopeless"]

def keyword_score(text):
    return sum(1 for k in keywords if k in text)

df["risk_score"] = df["clean_text"].apply(keyword_score)

In [11]:
def pronoun_count(text):
    return text.split().count("i")

df["pronoun_count"] = df["clean_text"].apply(pronoun_count)

In [12]:
df_final = df[[
    "clean_text",
    "label",
    "length",
    "risk_score",
    "pronoun_count"
]]

In [13]:
df_final["label"] = df_final["label"].map({
    "non-suicide": 0,
    "suicide": 1
})

In [14]:
df_final.to_csv("suicide-watch/suicide_watch_cleaned.csv", index=False)